### Problem Set "Portfolio Sorts" for BSc students



### Documentation

## Import statments
Some common libraries are used throghout this script

## Parameters
Some parameters are set, notably: <br>
Folders that point to where the input data, the logfile and the results lie <br>
Parameters directly relevant to the portfolio sorts algorithm, e.g. start and end date, sorting type, sortvariable names, weighing type etc. <br>

## Description of data
This problem set comes with the following input data: <br>
SPXConstituentsPrices.parquet --> Prices of the SPX constituents <br>
RiskFreeRatesFiltered.parquet --> Risk free rates <br>
CRAMMeasures.parquet --> Risk measures (sortvariables) <br>

## Code
# calcMonthlyExcessReturn()
This function loads prices and risk free rates from the baseFolder, where the data has to be put <br>
It further calculates monthly excess returns for each securityid (identifyer of stock) and offsets these returns by one month, to get the one month ahead excess return, which will later be the outcome variable for the portfolio sorts <br>
Input: SPXConstituentsPrices.parquet, RiskFreeRatesFiltered.parquet <br>
Output: RiskFreeRatesFilteredMonthly.parquet, SPXConstituentsPricesPrepared.parquet <br>

# mergeData()
This function merges the CRAM data (measures/sortvariables) and the recently calculated one month ahead excess returns together, based on the identifiyer SecurityID <br>
Input: SPXConstituentsPricesPrepared.parquet, CRAMMeasures.parquet <br>
Output: MergedInputData.parquet <br>

# getSortingDates()
This function determines the first available dates each month, which will later be used for sorting and filters the input data for relevant dates and columns <br>
Input: MergedInputData.parquet <br>
Output: MergedInputDataFiltered.parquet, SortingDates.parquet <br>

# buildSortedPortfolios()
This function performs the sorting algorithm. It takes all relevant parameters (variables) defined above, the input data and sorting dates. <br>
It iterates over each date in the date chunk <br>
For each date it collects information on missing data <br>
Breakpoints based on the sortvariable(s) are calculated. <br>
Instruments are sorted into portfolios based on these breakpoints <br>
Optionally, portfolios are filtered (excluded) <br>
Weights of each stock within each portfolio (marketcap or equally weighted) and the average outcome variable per portfolio are calculated <br>
the results are saved to pkl files <br>
Input: MergedInputDataFiltered.parquet, SortingDates.parquet <br>
Output: sortedPortfolioResults.pkl, sortedOutcomeResults.pkl, numberOfStocks.pkl, breakpointsDict.pkl, missingDataDict.pkl, missingRebalancingDatesList.pkl <br>

# convertOutcome()
This function converts the sortedOutcomeResults.pkl into a table and saves it into a parquet file. <br>
Input: sortedOutcomeResults.pkl <br>
Output: PortfoliosOutcome.parquet <br>

# calcDiffPortfolios()
This function calculates the diff portfolios based on the sorted portfolios depending on the type of sorting (univariate, bivariate) <br>
Input: PortfoliosOutcome.parquet <br>
Output: PortfoliosOutcomePlusDiff.parquet <br>

# performStatTest()
This function performs calculates the time series average and performes statistical tests on the sorted portfolio time series average outcome values. <br>
Input: PortfoliosOutcomePlusDiff.parquet <br>
Output: PortfoliosOutcomePlusDiffTest.parquet <br>

In [ ]:
# Import libraries
import os
import time
import pickle
import pandas as pd
import numpy as np
from loguru import logger
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

# Directories directly relevant to this script
baseFolder = '/Users/flo/Documents/Uni/FDS/FDS_py/PS4/Data1' # Project folder, relevant data is stored here
logFilePath = baseFolder + '/logfilePortfolioSortsPipeline.log' # The log file goes here
resultsPath = baseFolder + '/solution_Univariate' # Solutions go here

# Parameters
sortingType = "Univariate" # "Univariate", "BivariateDependent" or "BivariateIndependent"
sortVariable1 = "netis_at" # name of the first sorting variable
sortVariable2 = "market_equity" # name of the second sorting variable
sortPercentilesVar1 = [30, 70] # percentiles for the split of sort variable 1
sortPercentilesVar2 = [30, 70] # percentiles for the split of sort variable 2
weightType = "MarketCapWeighted" # weighting scheme within the portfolios, "MarketCapWeighted" or "EquallyWeighted"

testmode = False # if test mode is set to true, the sorting of portfolios will only be applied to a small number of dates
log_results = False # Variable to control whether the results of the portfolio sorting are printed in the log

# FIXED PARAMETERS - DO NOT CHANGE
startDate = pd.to_datetime('01.01.2012', format='%d.%m.%Y') # start date for the procedure
endDate = pd.to_datetime('31.12.2021', format='%d.%m.%Y') # end date for the procedure
marketCapColumn = "MCap" # name of the column that contains the market capitalization
outcomeVariable = "return_month_ahead_excess" # name of the outcome variable
filterPortfolios = None # Optional parameter for filtering out sorted portfolios by index
backtrackingDays = 0 # Variable to handle the number of days of backtracking in case of no measure data for portfolio sorts, values other than 0 are not supported in this code version
lags = 6 # the number of lags for the Newey-West adjustment for the statistical tests

# class that handles all logic
class PortfolioSortsPipeline():

    def __init__(self):
        self.start_time = None
        self.end_time = None

    def start(self):
        # Set up logging
        self.start_time = time.time()
        if os.path.exists(logFilePath):
            os.remove(logFilePath)
        logger.add(logFilePath, level='INFO')
        logger.info(f"Start Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(self.start_time))}")
        logger.info('Start portfolio sorts pipeline...')

    def clearSolutionsFolder(self):
        logger.info(f'Clearing solution folder at {resultsPath}...')
        if os.path.exists(resultsPath):
            for filename in os.listdir(resultsPath):
                file_path = os.path.join(resultsPath, filename)
                try:
                    if os.path.isfile(file_path) or os.path.islink(file_path):
                        os.unlink(file_path)
                    elif os.path.isdir(file_path):
                        os.rmdir(file_path)
                except Exception as e:
                    logger.error(f'Failed to delete {file_path}. Reason: {e}')
        else:
            os.makedirs(resultsPath)

    def end(self):
        self.end_time = time.time()
        duration_seconds = self.end_time - self.start_time
        duration_formatted = time.strftime('%H:%M:%S', time.gmtime(duration_seconds))
        logger.info('Portfolio sorts pipeline done!')
        logger.info(f"End Time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(self.end_time))}")
        logger.info(f'Duration: {duration_formatted}')

    def calcMonthlyExcessReturns(self):
        # File Paths
        prices_filepath = baseFolder + '/SPXConstituentsPrices.parquet'
        riskfree_filepath = baseFolder + '/RiskFreeRatesFiltered.parquet'
        riskfree_output = resultsPath + "/RiskFreeRatesFilteredMonthly.parquet"
        prices_output = resultsPath + "/SPXConstituentsPricesPrepared.parquet"

        # Load Prices and Risk-Free Rates
        pricesData = pd.read_parquet(prices_filepath)
        riskFreeRatesData = pd.read_parquet(riskfree_filepath)

        pricesData['date'] = pd.to_datetime(pricesData['date'])
        riskFreeRatesData['date'] = pd.to_datetime(riskFreeRatesData['date'])

        ########## STUDENT TASK START ##########
        # Produce two output files:
        #   (1) riskFreeRatesData -> riskfree_output (parquet, index=False)
        #   (2) pricesData        -> prices_output    (parquet, index=False)
        # pricesData must carry one column called 'return_month_ahead_excess'
        # holding each row's next-month excess return for the SPX constituent.

        # YOUR CODE HERE
        riskFreeRatesData = riskFreeRatesData[
            riskFreeRatesData['daystomaturity'] == 30
        ].copy()
        riskFreeRatesData['YearMonth'] = riskFreeRatesData['date'].dt.to_period('M')
        riskFreeRatesData = (
            riskFreeRatesData.sort_values('date')
                             .groupby('YearMonth', as_index=False)
                             .first()
        )
        riskFreeRatesData.to_parquet(riskfree_output, index=False)

        pricesData = pricesData.sort_values(['securityid', 'date']).copy()
        pricesData['MCap'] = pricesData['closeprice'] * pricesData['sharesoutstanding']
        pricesData['YearMonth'] = pricesData['date'].dt.to_period('M')

        monthly = (
            pricesData.groupby(['securityid', 'YearMonth'], as_index=False)
                      .agg(monthly_return=('totalreturn',
                                           lambda r: (1.0 + r).prod() - 1.0),
                           MCap=('MCap', 'last'))
        )
        monthly = monthly.merge(
            riskFreeRatesData[['YearMonth', 'yld_pct_monthly']],
            on='YearMonth', how='left',
        )
        monthly['monthly_excess'] = monthly['monthly_return'] - monthly['yld_pct_monthly']

        monthly = monthly.sort_values(['securityid', 'YearMonth'])
        monthly['return_month_ahead_excess'] = (
            monthly.groupby('securityid')['monthly_excess'].shift(-1)
        )

        # Align dates to calendar month-end so they match equityZoo's loctimestamp
        monthly['date'] = monthly['YearMonth'].dt.to_timestamp() + pd.offsets.MonthEnd(0)
        pricesData = monthly[
            ['securityid', 'date', 'MCap', 'return_month_ahead_excess']
        ].reset_index(drop=True)
        pricesData.to_parquet(prices_output, index=False)
        ########## STUDENT TASK END ##########

        logger.info(f"Processed risk free rates saved to {riskfree_output}")
        logger.info(f"Processed data saved to {prices_output}")

    def mergeData(self):
        # File paths for the data
        prices_filepath = resultsPath + "/SPXConstituentsPricesPrepared.parquet"
        cram_filepath = baseFolder + "/equityZooMeasures.parquet"
        merged_output = resultsPath + "/MergedInputData.parquet"

        # Load the data
        pricesData = pd.read_parquet(prices_filepath)
        cramData = pd.read_parquet(cram_filepath)

        ########## STUDENT TASK START ##########
        # Produce 'mergedData' and save it to merged_output (parquet,
        # index=False). Each row must carry the CRAM measure values for a
        # given (security, date) plus that security's next-month excess
        # return from pricesData.

        # YOUR CODE HERE
        pricesData['date'] = pd.to_datetime(pricesData['date'])
        cramData['loctimestamp'] = pd.to_datetime(cramData['loctimestamp'])

        mergedData = cramData.merge(
            pricesData[['securityid', 'date', 'MCap', 'return_month_ahead_excess']],
            left_on=['SecurityID', 'loctimestamp'],
            right_on=['securityid', 'date'],
            how='inner',
        ).drop(columns=['securityid', 'date'])

        ########## STUDENT TASK END ##########

        mergedData.to_parquet(merged_output, index=False)
        logger.info(f"Merged data saved to {merged_output}")

    def getSortingDates(self):

        # File paths
        merged_filepath = resultsPath + "/MergedInputData.parquet"
        filtered_output = resultsPath + "/MergedInputDataFiltered.parquet"
        sorting_dates_output = resultsPath + "/SortingDates.parquet"

        # Load merged data
        mergedData = pd.read_parquet(merged_filepath)

        mergedData['loctimestamp'] = pd.to_datetime(mergedData['loctimestamp'])

        ########## STUDENT TASK START ##########
        # Produce TWO objects:
        #   first_of_month_dates : a pd.Series of the first available trading
        #                          day in each calendar month
        #   filteredData         : mergedData restricted to rows whose
        #                          loctimestamp is one of those days

        # YOUR CODE HERE
        mergedData = mergedData.sort_values('loctimestamp')
        month_period = mergedData['loctimestamp'].dt.to_period('M')
        first_of_month_dates = (
            mergedData.groupby(month_period)['loctimestamp']
                      .first()
                      .reset_index(drop=True)
        )
        first_of_month_dates = first_of_month_dates[
            (first_of_month_dates >= startDate) & (first_of_month_dates <= endDate)
        ].reset_index(drop=True)
        filteredData = mergedData[
            mergedData['loctimestamp'].isin(first_of_month_dates)
        ]

        ########## STUDENT TASK END ##########

        filteredData = filteredData.reset_index(drop=True)
        first_of_month_dates = first_of_month_dates.reset_index(drop=True)

        # Save the filtered dataframe
        filteredData.to_parquet(filtered_output, index=False)

        # Save the list of first-of-month dates
        first_of_month_dates.to_frame(name="first_of_month").to_parquet(sorting_dates_output, index=False)

        logger.info(f"Filtered dataset saved to {filtered_output}")
        logger.info(f"Sorting dates saved to {sorting_dates_output}")

    def buildSortedPortfolios(self):
        # File paths
        instrument_data_path = resultsPath + "/MergedInputDataFiltered.parquet"
        instrumentData = pd.read_parquet(instrument_data_path)
        sorting_dates_path = resultsPath + "/SortingDates.parquet"
        sortingDates = pd.read_parquet(sorting_dates_path)
        sortingDates = sortingDates["first_of_month"].tolist() # convert to list
        if testmode:
            sortingDates = sortingDates[0:5] # Short list for testing
        logger.info(f'Starting calculation of sorted portfolios: {sortingType}')
        logger.info(f'Sorting variable 1: {sortVariable1}, sorting variable 2: {sortVariable2}, outcome variable: {outcomeVariable}, percentiles variable 1: {sortPercentilesVar1}, percentiles variable 2: {sortPercentilesVar2}')
        logger.info(f'Weighting: {weightType}, filter for portfolios: {filterPortfolios}, maximum possible backtracking: {backtrackingDays} days')
        portfolioWeightsDict = {} # Key: portfolioID, Value: {date: weights}
        outcomeResultsDict = {} # Key: portfolioID, Value: {date: averageOutcome}
        numberOfStocksDict = {} # Number of stocks per portfolio
        breakpointsDict = {} # Breakpoints per date
        missingDataDict = {} # Key: date, Value: {errortype: count}
        missingRebalancingDatesList = [] # List containing missing rebalancing dates
        constituents = instrumentData['instrumentid'].unique().tolist()

        def countNaNs(df, colname): # Function to handle possible duplicate column names when counting errors
            col = df[colname]
            if isinstance(col, pd.DataFrame): # If column name, i.e. variable, comes up more than once
                return col.iloc[:, 0].isna().sum()
            return col.isna().sum()
        
        for date in sortingDates: # Iterate over each date and perform sorting
            workingDate = date
            validDataFound = False
            backtrackAttempts = 0
            missingData = {date: {"instruments_missing": 0, "sortVariable1_nan": 0, "sortVariable2_nan": 0, "marketCap_nan": 0, "outcomeVar_nan": 0, "portfolio_insufficient": 0}}
            if sortingType == "Univariate":
                # This date's slice of (instrument, sort variable, mcap, outcome)
                dateDataframe = instrumentData[
                    (instrumentData['loctimestamp'] == workingDate) &
                    (instrumentData['instrumentid'].isin(constituents))
                ][['instrumentid', sortVariable1, marketCapColumn, outcomeVariable]]
                validDataframe = dateDataframe.dropna(
                    subset=[sortVariable1, marketCapColumn, outcomeVariable]
                ).copy()
                if validDataframe.empty:
                    missingRebalancingDatesList.append(date)
                    continue

                # Bookkeeping for missing data (DO NOT MODIFY)
                missingIDs = set(constituents) - set(dateDataframe['instrumentid'])
                missingData[date]["instruments_missing"] += len(missingIDs)
                missingData[date]["sortVariable1_nan"] += int(countNaNs(dateDataframe, sortVariable1))
                missingData[date]["marketCap_nan"]     += int(countNaNs(dateDataframe, marketCapColumn))
                missingData[date]["outcomeVar_nan"]    += int(countNaNs(dateDataframe, outcomeVariable))
                missingDataDict[date] = missingData[date]

                ########## STUDENT TASK START ##########
                # Implement the per-date univariate sort. Populate three locals:
                #
                #   breakpoints      : list of cut points used to bucket
                #                      instruments by sortVariable1
                #
                #   portfolioWeights : dict
                #       key   = f"{sortVariable1}_{k}" for k in 1..n_portfolios
                #       value = {instrumentID -> weight}
                #
                #   portfolioOutcome : dict
                #       key   = same f"{sortVariable1}_{k}" string
                #       value = the portfolio's weighted-average outcome
                #
                # The weighting scheme is selected by the module-level
                # parameter weightType. The optional filterPortfolios parameter,
                # if truthy, is a list of portfolio indices (1-based) to keep;
                # otherwise keep all.

                # YOUR CODE HERE
                sort1Values = validDataframe[sortVariable1].to_numpy()
                breakpoints = list(np.percentile(sort1Values, sortPercentilesVar1))
                n_portfolios = len(sortPercentilesVar1) + 1
                edges = [-np.inf] + list(breakpoints) + [np.inf]

                validDataframe['bucket'] = pd.cut(
                    validDataframe[sortVariable1],
                    bins=edges,
                    labels=list(range(1, n_portfolios + 1)),
                    include_lowest=True,
                )

                portfolioWeights = {}
                portfolioOutcome = {}
                for k in range(1, n_portfolios + 1):
                    if filterPortfolios and k not in filterPortfolios:
                        continue
                    portfolioID = f"{sortVariable1}_{k}"
                    bucket_df = validDataframe[validDataframe['bucket'] == k]
                    if bucket_df.empty:
                        continue
                    if weightType == "MarketCapWeighted":
                        w_series = bucket_df[marketCapColumn] / bucket_df[marketCapColumn].sum()
                    else:
                        w_series = pd.Series(1.0 / len(bucket_df), index=bucket_df.index)
                    weights = dict(zip(bucket_df['instrumentid'].tolist(), w_series.tolist()))
                    weighted_outcome = float(
                        (w_series.values * bucket_df[outcomeVariable].values).sum()
                    )
                    portfolioWeights[portfolioID] = weights
                    portfolioOutcome[portfolioID] = weighted_outcome

                ########## STUDENT TASK END ##########

                # Fold per-date results into the global accumulators (DO NOT MODIFY)
                for portfolioID, weightStructure in portfolioWeights.items():
                    portfolioWeightsDict.setdefault(portfolioID, {})[date] = weightStructure
                for portfolioID, outcome in portfolioOutcome.items():
                    outcomeResultsDict.setdefault(portfolioID, {})[date] = outcome
                breakpointsDict.setdefault(date, {})[sortVariable1] = breakpoints

            elif sortingType in ["BivariateDependent", "BivariateIndependent"]:
                while not validDataFound and backtrackAttempts <= backtrackingDays:
                    # Filter instrumentData to current date and relevant instrument IDs
                    dateDataframe = instrumentData[
                        (instrumentData['instrumentid'].isin(constituents)) &
                        (instrumentData['loctimestamp'] == workingDate)
                    ][['instrumentid', sortVariable1, sortVariable2, marketCapColumn, outcomeVariable]]
                    validDataframe = dateDataframe.dropna(subset=[sortVariable1, sortVariable2, marketCapColumn, outcomeVariable]).copy() # Drop rows with any NaNs in relevant columns
                    # Step 1: Determine values of the sort variables, if available
                    sort1Values = validDataframe[sortVariable1].to_numpy() # Collect values of sortVariable1
                    sort2Values = validDataframe[sortVariable2].to_numpy() # Collect values of sortVariable2
                    if len(sort1Values) > 0 and len(sort2Values) > 0:
                        validDataFound = True
                    else:
                        logger.warning(f"No valid CRAM data found for {workingDate}. Backtracking to the previous date.")
                        backtrackAttempts += 1
                        workingDate = workingDate - pd.Timedelta(days=1)
                if backtrackAttempts > backtrackingDays:
                    logger.error(f"ERROR Failed to find valid CRAM data for {date} after {backtrackingDays} attempts.")
                    missingRebalancingDatesList.append(date)
                    continue # Stop searching and log error for the original date.
                # Count how many expected instruments were not found for the date
                missingIDs = set(constituents) - set(dateDataframe['instrumentid'])
                missingData[date]["instruments_missing"] += len(missingIDs)
                # Count missing values in each column
                missingData[date]["sortVariable1_nan"] += int(countNaNs(dateDataframe, sortVariable1))
                missingData[date]["sortVariable2_nan"] += int(countNaNs(dateDataframe, sortVariable2))
                missingData[date]["marketCap_nan"]     += int(countNaNs(dateDataframe, marketCapColumn))
                missingData[date]["outcomeVar_nan"]    += int(countNaNs(dateDataframe, outcomeVariable))
                missingDataDict[date] = missingData[date]

                if sortingType == "BivariateDependent":
                    ########## STUDENT TASK START ##########
                    # Implement a DEPENDENT bivariate sort for this date.
                    # Produce portfolioWeights and portfolioOutcome with the
                    # same structure as the univariate case, but keyed by
                    #   f"{sortVariable1}_{i}_{sortVariable2}_{j}"
                    # Record this date's breakpoints in breakpointsDict[date].
                    #
                    # If a portfolio cannot be formed, append `date` to
                    # missingRebalancingDatesList, mark
                    # missingData[date]["portfolio_insufficient"] += 1, and
                    # `continue`.

                    # YOUR CODE HERE
                    breakpoints1 = list(np.percentile(sort1Values, sortPercentilesVar1))
                    n1 = len(sortPercentilesVar1) + 1
                    n2 = len(sortPercentilesVar2) + 1
                    edges1 = [-np.inf] + list(breakpoints1) + [np.inf]

                    validDataframe['bucket1'] = pd.cut(
                        validDataframe[sortVariable1],
                        bins=edges1,
                        labels=list(range(1, n1 + 1)),
                        include_lowest=True,
                    )

                    portfolioWeights = {}
                    portfolioOutcome = {}
                    breakpoints2_collected = {}
                    date_failed = False

                    for i in range(1, n1 + 1):
                        sub_i = validDataframe[validDataframe['bucket1'] == i].copy()
                        if sub_i.empty:
                            missingRebalancingDatesList.append(date)
                            missingData[date]["portfolio_insufficient"] += 1
                            date_failed = True
                            break
                        bp2 = list(np.percentile(sub_i[sortVariable2].to_numpy(), sortPercentilesVar2))
                        breakpoints2_collected[i] = bp2
                        edges2 = [-np.inf] + list(bp2) + [np.inf]
                        sub_i['bucket2'] = pd.cut(
                            sub_i[sortVariable2],
                            bins=edges2,
                            labels=list(range(1, n2 + 1)),
                            include_lowest=True,
                        )
                        for j in range(1, n2 + 1):
                            portfolioID = f"{sortVariable1}_{i}_{sortVariable2}_{j}"
                            bucket_df = sub_i[sub_i['bucket2'] == j]
                            if bucket_df.empty:
                                missingRebalancingDatesList.append(date)
                                missingData[date]["portfolio_insufficient"] += 1
                                date_failed = True
                                break
                            if weightType == "MarketCapWeighted":
                                w_series = bucket_df[marketCapColumn] / bucket_df[marketCapColumn].sum()
                            else:
                                w_series = pd.Series(1.0 / len(bucket_df), index=bucket_df.index)
                            weights = dict(zip(bucket_df['instrumentid'].tolist(), w_series.tolist()))
                            weighted_outcome = float(
                                (w_series.values * bucket_df[outcomeVariable].values).sum()
                            )
                            portfolioWeights[portfolioID] = weights
                            portfolioOutcome[portfolioID] = weighted_outcome
                        if date_failed:
                            break

                    breakpointsDict[date] = {sortVariable1: breakpoints1, sortVariable2: breakpoints2_collected}
                    if date_failed:
                        continue

                    ########## STUDENT TASK END ##########

                    # Fold per-date results into the global accumulators (DO NOT MODIFY)
                    missingDataDict[date] = missingData[date]
                    for portfolioID, weightStructure in portfolioWeights.items():
                        portfolioWeightsDict.setdefault(portfolioID, {})[date] = weightStructure
                    for portfolioID, outcome in portfolioOutcome.items():
                        outcomeResultsDict.setdefault(portfolioID, {})[date] = outcome

                elif sortingType == "BivariateIndependent":
                    ########## STUDENT TASK START ##########
                    # Implement an INDEPENDENT bivariate sort for this date.
                    # Produce portfolioWeights and portfolioOutcome with the
                    # same structure as the dependent case.
                    # Record breakpointsDict[date].
                    #
                    # If any resulting portfolio ends up empty, append `date`
                    # to missingRebalancingDatesList, mark
                    # missingData[date]["portfolio_insufficient"] += 1, and
                    # `continue`.

                    # YOUR CODE HERE
                    breakpoints1 = list(np.percentile(sort1Values, sortPercentilesVar1))
                    breakpoints2 = list(np.percentile(sort2Values, sortPercentilesVar2))
                    n1 = len(sortPercentilesVar1) + 1
                    n2 = len(sortPercentilesVar2) + 1
                    edges1 = [-np.inf] + list(breakpoints1) + [np.inf]
                    edges2 = [-np.inf] + list(breakpoints2) + [np.inf]

                    validDataframe['bucket1'] = pd.cut(
                        validDataframe[sortVariable1],
                        bins=edges1,
                        labels=list(range(1, n1 + 1)),
                        include_lowest=True,
                    )
                    validDataframe['bucket2'] = pd.cut(
                        validDataframe[sortVariable2],
                        bins=edges2,
                        labels=list(range(1, n2 + 1)),
                        include_lowest=True,
                    )

                    portfolioWeights = {}
                    portfolioOutcome = {}
                    date_failed = False
                    for i in range(1, n1 + 1):
                        for j in range(1, n2 + 1):
                            portfolioID = f"{sortVariable1}_{i}_{sortVariable2}_{j}"
                            bucket_df = validDataframe[
                                (validDataframe['bucket1'] == i)
                                & (validDataframe['bucket2'] == j)
                            ]
                            if bucket_df.empty:
                                missingRebalancingDatesList.append(date)
                                missingData[date]["portfolio_insufficient"] += 1
                                date_failed = True
                                break
                            if weightType == "MarketCapWeighted":
                                w_series = bucket_df[marketCapColumn] / bucket_df[marketCapColumn].sum()
                            else:
                                w_series = pd.Series(1.0 / len(bucket_df), index=bucket_df.index)
                            weights = dict(zip(bucket_df['instrumentid'].tolist(), w_series.tolist()))
                            weighted_outcome = float(
                                (w_series.values * bucket_df[outcomeVariable].values).sum()
                            )
                            portfolioWeights[portfolioID] = weights
                            portfolioOutcome[portfolioID] = weighted_outcome
                        if date_failed:
                            break

                    breakpointsDict[date] = {sortVariable1: breakpoints1, sortVariable2: breakpoints2}
                    if date_failed:
                        continue

                    ########## STUDENT TASK END ##########

                    # Fold per-date results into the global accumulators (DO NOT MODIFY)
                    missingDataDict[date] = missingData[date]
                    for portfolioID, weightStructure in portfolioWeights.items():
                        portfolioWeightsDict.setdefault(portfolioID, {})[date] = weightStructure
                    for portfolioID, outcome in portfolioOutcome.items():
                        outcomeResultsDict.setdefault(portfolioID, {})[date] = outcome

        # Sort portfolio results by date ascending for each portfolio
        sortedPortfolioWeightsDict = {
            portfolioID: dict(sorted(weightsByDate.items()))
            for portfolioID, weightsByDate in portfolioWeightsDict.items()
            }
        # Sort outcome results by date ascending for each portfolio
        sortedOutcomeResultsDict = {
            portfolioID: dict(sorted(outcomeByDate.items()))
            for portfolioID, outcomeByDate in outcomeResultsDict.items()
            }
        # Sort missingData by date ascending
        missingDataDict = dict(sorted(missingDataDict.items(), key=lambda item: item[0]))
        # Sort breakpoints by date ascending
        breakpointsDict = dict(sorted(breakpointsDict.items(), key=lambda item: item[0]))
        # Count number of stocks
        for portfolioID, portfolioWeights in sortedPortfolioWeightsDict.items():
            if portfolioID not in numberOfStocksDict:
                numberOfStocksDict[portfolioID] = {}
            for date, weights in portfolioWeights.items():
                numberOfStocksDict[portfolioID][date] = len(weights)
        
        # PRINT RESULTS TO LOG
        if log_results:
            # logger.info("------Portfolios:-------")
            # for portfolio, weightStructure in sortedPortfolioWeightsDict.items():
            #     logger.info(f'Portfolio {portfolio}:')
            #     for date, weights in weightStructure.items():
            #         logger.info(f'Date {date}: Something')
            logger.info("------Outcome:-------")
            for portfolio, outcomeStruct in sortedOutcomeResultsDict.items():
                logger.info(f'Portfolio {portfolio}:')
                for date, outcome in outcomeStruct.items():
                    logger.info(f'Date {date}: {outcome}')
            logger.info("------Missing Data:------")
            for date, dataInfo in missingDataDict.items():
                logger.info(date)
                logger.info(dataInfo)
            logger.info("------Missing rebalancing dates:------")
            logger.info(missingRebalancingDatesList)
            logger.info("------Number of stocks:------")
            for portfolio, nstocksStruct in numberOfStocksDict.items():
                logger.info(f'Portfolio {portfolio}:')
                for date, nstocks in nstocksStruct.items():
                    logger.info(f'Date {date}: {nstocks}')
            logger.info("------Breakpoints:------")
            for date, struct in breakpointsDict.items():
                logger.info(f'date: {date}')
                for var, bps in struct.items():
                    logger.info(f'variable: {var}')
                    logger.info(f'breakpoints: {bps}')
        # Dictionary of variables to save
        variables_to_save = {
            "sortedPortfolioResults": sortedPortfolioWeightsDict,
            "sortedOutcomeResults": sortedOutcomeResultsDict,
            "numberOfStocks": numberOfStocksDict,
            "breakpointsDict": breakpointsDict,
            "missingDataDict": missingDataDict,
            "missingRebalancingDatesList": missingRebalancingDatesList
        }
        # Loop through dictionary and save each variable as a .pkl file
        for var_name, var_value in variables_to_save.items():
            file_path = os.path.join(resultsPath, f"{var_name}.pkl")
            with open(file_path, "wb") as f:
                pickle.dump(var_value, f)
            logger.info(f"Saved {var_name} to {file_path}")
        logger.info('Completed calculation of sorted portfolios.')

    def convertOutcome(self):
        # Path to the input pickle file
        filtered_output = resultsPath + "/sortedOutcomeResults.pkl"
        
        # Load the sortedOutcomeResults from pickle
        with open(filtered_output, 'rb') as f:
            sortedOutcomeResults = pickle.load(f)
        
        # Create a list to store all the dates (we assume the dates are the same for all portfolios)
        dates = list(next(iter(sortedOutcomeResults.values())).keys())

        # Create a dictionary where keys are portfolio names and values are the corresponding outcome values for each date
        portfolio_data = {}
        
        for portfolio, outcomeStruct in sortedOutcomeResults.items():
            # Create a list of outcomes for the given portfolio, aligned with the dates
            portfolio_data[portfolio] = [outcomeStruct.get(date, None) for date in dates]

        # Convert the dictionary into a DataFrame
        outcome_df = pd.DataFrame(portfolio_data, index=dates)
        
        # Reset index so 'date' becomes a column in the DataFrame
        outcome_df.reset_index(inplace=True)
        outcome_df.rename(columns={'index': 'date'}, inplace=True)

        # Save the DataFrame to a Parquet file
        outcome_df.to_parquet(resultsPath + "/PortfoliosOutcome.parquet", index=False)
        logger.info("PortfoliosOutcome.parquet has been saved.")

    def calcDiffPortfolios(self):
        # Load the PortfoliosOutcome.parquet file
        portfolios_outcome_filepath = resultsPath + "/PortfoliosOutcome.parquet"
        outcome_df = pd.read_parquet(portfolios_outcome_filepath)

        if sortingType == "Univariate":
            # Determine the portfolio names for the highest and lowest portfolios based on sortVariable1
            high_portfolio = f"{sortVariable1}_{len(sortPercentilesVar1) + 1}"  # Highest portfolio
            low_portfolio = f"{sortVariable1}_1"  # Lowest portfolio
            
            # Check if the high and low portfolios exist in the columns
            if high_portfolio not in outcome_df.columns or low_portfolio not in outcome_df.columns:
                logger.error(f"Portfolio columns {high_portfolio} or {low_portfolio} not found in the DataFrame.")
                return
            
            # Add the new column: sortVariable1_Diff (difference between high and low portfolio)
            outcome_df[f"{sortVariable1}_Diff"] = outcome_df[high_portfolio] - outcome_df[low_portfolio]

        elif sortingType in ("BivariateDependent", "BivariateIndependent"):
            logger.info(f"Calculating generalized Diff and Average logic for bivariate sorting: {sortingType}")
            
            # 1. Determine group lengths dynamically from notebook parameters
            n_g1 = len(sortPercentilesVar1) + 1 # Number of portfolios along Sort Variable 1 (rows)
            n_g2 = len(sortPercentilesVar2) + 1 # Number of portfolios along Sort Variable 2 (columns)
            
            # Helper to generate the exact base portfolio string keys
            def get_col(i, j):
                return f"{sortVariable1}_{i}_{sortVariable2}_{j}"

            # --- A. ROW-WISE MARGINAL CALCULATIONS (Across Sort Variable 2 columns) ---
            # For each category of Var 1, compute the high-low difference and row average
            for i in range(1, n_g1 + 1):
                row_cols = [get_col(i, j) for j in range(1, n_g2 + 1)]
                
                # Diff across Var 2 = High - Low (e.g., group n_g2 - group 1)
                outcome_df[f"{sortVariable1}_{i}_{sortVariable2}_Diff"] = outcome_df[get_col(i, n_g2)] - outcome_df[get_col(i, 1)]
                # Avg across Var 2 = Row Mean
                outcome_df[f"{sortVariable1}_{i}_{sortVariable2}_Avg"] = outcome_df[row_cols].mean(axis=1)

            # --- B. COLUMN-WISE MARGINAL CALCULATIONS (Down Sort Variable 1 rows) ---
            # For each category of Var 2, compute the high-low difference and column average
            for j in range(1, n_g2 + 1):
                col_cols = [get_col(i, j) for i in range(1, n_g1 + 1)]
                
                # Diff down Var 1 = High - Low (e.g., group n_g1 - group 1)
                outcome_df[f"{sortVariable1}_Diff_{sortVariable2}_{j}"] = outcome_df[get_col(n_g1, j)] - outcome_df[get_col(1, j)]
                # Avg down Var 1 = Column Mean
                outcome_df[f"{sortVariable1}_Avg_{sortVariable2}_{j}"] = col_cols_df = outcome_df[col_cols].mean(axis=1)

            # --- C. CROSS-OVER INTERSECTIONS (The Matrix Outer Corners) ---
            # 1. Sort Variable 1 Diff pooled across Sort Variable 2 Average
            var1_diff_cols = [f"{sortVariable1}_Diff_{sortVariable2}_{j}" for j in range(1, n_g2 + 1)]
            outcome_df[f"{sortVariable1}_Diff_{sortVariable2}_Avg"] = outcome_df[var1_diff_cols].mean(axis=1)
            
            # 2. Sort Variable 2 Diff pooled across Sort Variable 1 Average
            var2_diff_cols = [f"{sortVariable1}_{i}_{sortVariable2}_Diff" for i in range(1, n_g1 + 1)]
            outcome_df[f"{sortVariable1}_Avg_{sortVariable2}_Diff"] = outcome_df[var2_diff_cols].mean(axis=1)

            # 3. Grand Total Sample Mean Portfolio (Average of Average)
            all_base_cols = [get_col(i, j) for i in range(1, n_g1 + 1) for j in range(1, n_g2 + 1)]
            outcome_df[f"{sortVariable1}_Avg_{sortVariable2}_Avg"] = outcome_df[all_base_cols].mean(axis=1)

            # 4. Difference-in-Difference Portfolio (Double Diff)
            # Implements the canonical (D - C) - (B - A) algebraic structure from Table 5.12
            A = outcome_df[get_col(1, 1)]         # Low Var1, Low Var2
            B = outcome_df[get_col(1, n_g2)]      # Low Var1, High Var2
            C = outcome_df[get_col(n_g1, 1)]      # High Var1, Low Var2
            D = outcome_df[get_col(n_g1, n_g2)]   # High Var1, High Var2
            
            outcome_df[f"{sortVariable1}_Diff_{sortVariable2}_Diff"] = (D - C) - (B - A)

        # Save the resulting DataFrame to a new Parquet file
        output_filepath = resultsPath + "/PortfoliosOutcomePlusDiff.parquet"
        outcome_df.to_parquet(output_filepath, index=False)
        
        logger.info(f"Generalized resulting DataFrame with structural columns saved to {output_filepath}")

    def performStatTest(self):
        # Load the PortfoliosOutcomePlusDiff.parquet file
        portfolios_outcome_filepath = resultsPath + "/PortfoliosOutcomePlusDiff.parquet"
        outcome_df = pd.read_parquet(portfolios_outcome_filepath)

        # Exclude the 'date' column and work with portfolio columns
        portfolio_columns = [col for col in outcome_df.columns if col != 'date']

        # Initialize a list to hold the statistics (Avg, StdErr, t-stat, p-value)
        stats_dict = {
            "Average": [],
            "StdErr": [],
            "t-stat": [],
            "p-value": []
        }

        for portfolio in portfolio_columns:
            timeseries = outcome_df[portfolio]

            ########## STUDENT TASK START ##########
            # For this portfolio's time series of returns, compute:
            #   avg     : the unconditional mean
            #   std_err : the standard error of the mean
            #   t_stat  : the Newey-West (HAC) t-statistic for mean = 0,
            #             using `lags` lags (a module-level parameter)
            #   p_value : the two-sided p-value associated with t_stat

            # YOUR CODE HERE
            ts = timeseries.dropna().to_numpy()
            if len(ts) < 2:
                avg = std_err = t_stat = p_value = float('nan')
            else:
                X = np.ones((len(ts), 1))
                model = OLS(ts, X).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
                avg = float(model.params[0])
                std_err = float(model.bse[0])
                t_stat = float(model.tvalues[0])
                p_value = float(model.pvalues[0])

            ########## STUDENT TASK END ##########

            stats_dict["Average"].append(avg)
            stats_dict["StdErr"].append(std_err)
            stats_dict["t-stat"].append(t_stat)
            stats_dict["p-value"].append(p_value)

        # Create a DataFrame from the statistics dictionary
        stats_df = pd.DataFrame(stats_dict, index=portfolio_columns)

        # Save the resulting DataFrame to a new Parquet file
        output_filepath = resultsPath + "/PortfoliosOutcomePlusDiffTest.parquet"
        stats_df.to_parquet(output_filepath, index=True)

        logger.info(f"Statistical test results saved to {output_filepath}")

#--------------------
#-----Execution------
#--------------------

# Create object and run all steps one after the other
ps = PortfolioSortsPipeline()
ps.start()
ps.clearSolutionsFolder()
ps.calcMonthlyExcessReturns()
ps.mergeData()
ps.getSortingDates()
ps.buildSortedPortfolios()
ps.convertOutcome()
ps.calcDiffPortfolios()
ps.performStatTest()
ps.end()



